# GPU part (Colab)

Step 1: check that VGGT reconstructs our table from 10 frames of video A.
Runtime > Change runtime type > T4 GPU, then run all cells.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "https://github.com/golovchits/visibility-aware-change-detection.git"
!git clone -q {REPO} /content/repo || echo "already cloned"
%cd /content/repo
# --no-deps: VGGT pins numpy<2, which would downgrade Colab's numpy. einops is the only missing dependency.
!pip install -q --no-deps git+https://github.com/facebookresearch/vggt.git
!pip install -q einops

## 10 frames of video A
Evenly spaced across the whole video. A stand-in until `frames.py` (step 2) picks sharp frames.

In [ ]:
import os
import cv2
import plotly.express as px

video = cv2.VideoCapture("data/before.MOV")
n_total = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
os.makedirs("/content/frames_A", exist_ok=True)

frame_paths = []
for i in range(10):
    index = i * (n_total - 1) // 9
    video.set(cv2.CAP_PROP_POS_FRAMES, index)
    ok, frame = video.read()
    assert ok, f"could not read frame {index}"
    path = f"/content/frames_A/{index:04d}.jpg"
    cv2.imwrite(path, frame)
    frame_paths.append(path)

# OpenCV stores colour as BGR, Plotly expects RGB.
px.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), title=f"last of {len(frame_paths)} frames, should be upright").show()

## VGGT
One forward pass gives every frame a camera, a depth map and a confidence map.
fp16 because the T4 has no bf16 support.

In [ ]:
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

model = VGGT.from_pretrained("facebook/VGGT-1B").to("cuda").eval()
images = load_and_preprocess_images(frame_paths).to("cuda")  # (frames, 3, H, W), values 0 to 1

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    predictions = model(images)

extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions["pose_enc"], images.shape[-2:])
depth = predictions["depth"][0].float().cpu().numpy()            # (frames, H, W, 1)
confidence = predictions["depth_conf"][0].float().cpu().numpy()  # (frames, H, W)
extrinsic = extrinsic[0].float().cpu().numpy()                   # (frames, 3, 4), camera from world
intrinsic = intrinsic[0].float().cpu().numpy()                   # (frames, 3, 3)
print("images", tuple(images.shape), "depth", depth.shape)

## Point cloud
Every pixel goes back into 3D using its depth and its camera.
Only the most confident half of the points is kept, then 100k are drawn at random so the plot stays fast.

In [ ]:
import numpy as np
import plotly.graph_objects as go

points = unproject_depth_map_to_point_map(depth, extrinsic, intrinsic)  # (frames, H, W, 3)
colours = images.permute(0, 2, 3, 1).cpu().numpy()                       # (frames, H, W, 3)

keep = confidence > np.median(confidence)
points, colours = points[keep], colours[keep]
pick = np.random.default_rng(0).choice(len(points), size=min(100_000, len(points)), replace=False)
points, colours = points[pick], colours[pick]

rgb = [f"rgb({r},{g},{b})" for r, g, b in (colours * 255).astype(int)]
fig = go.Figure(go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2],
                             mode="markers", marker=dict(size=1, color=rgb)))
fig.update_layout(scene=dict(aspectmode="data"), height=700)
fig.show()